# Toyota/Lexus Repair Assistant Using Retrieval-Augmented Generation
### Claudio Cela

This notebook implements and evaluates a retrieval-augmented generation system for Toyota and Lexus repair explanations. The knowledge base is built from service-manual content for a 2001 Toyota Tundra Limited, 4WD and a 2001 Lexus LS 430. The system is evaluated before and after retrieval is added to determine whether vehicle-specific context improves structured, plain-English responses.

## 1. RAG Approach

For this project, I use retrieval-augmented generation instead of full fine-tuning. The goal is not to make the model memorize Toyota or Lexus repair procedures. Instead, the assistant retrieves the most relevant service-manual pages for a specific vehicle, diagnostic code, or symptom and then explains that information in a safer and easier-to-understand format. This is a better fit for automotive repair because the same symptom can mean different things depending on the model year, engine, trim, diagnostic code, and source document. A model answering from memory alone can easily mix repair steps, torque specifications, part locations, and safety concerns across vehicles.

My earlier model testing supports this approach. Qwen/Qwen2.5-3B-Instruct produced the strongest responses overall and gave the clearest structured repair explanations, but retrieval quality remained the largest weakness. When the retrieved context was relevant, the model had a better chance of producing a useful answer. When the context was unrelated, even the strongest model sometimes relied on general automotive knowledge. I therefore use Qwen/Qwen2.5-3B-Instruct as the generation model and focus the main improvements on Toyota and Lexus service-manual chunks, metadata filtering, and multiple retrieval methods. The model is loaded with GPU acceleration and 4-bit quantization to keep the evaluation stable on my local machine.

The main drawback of RAG is that the answer is only as good as the retrieved context. If the retriever returns an unrelated EVAP page for an overheating question, the model may give an incomplete answer or add unsupported general knowledge. The advantages still outweigh this limitation because the pipeline can be improved through better chunking, metadata, exact DTC matching, and additional source documents without retraining the generator. This also matches the intended purpose of the assistant: supporting repair understanding through source-grounded explanations rather than replacing a mechanic or guessing from memory.

### Sources

- Lewis et al. (2020), *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: https://arxiv.org/abs/2005.11401
- Friel et al. (2024), *RAGBench: Explainable Benchmark for Retrieval-Augmented Generation Systems*: https://arxiv.org/abs/2407.11005
- Niu et al. (2024), *RAGTruth: A Hallucination Corpus for Developing Trustworthy Retrieval-Augmented Language Models*: https://aclanthology.org/2024.acl-long.585/
- Chen et al. (2024), *Benchmarking Large Language Models in Retrieval-Augmented Generation*: https://doi.org/10.1609/aaai.v38i16.29728

## 2. Baseline Evaluation

Before implementing retrieval, I evaluate Qwen/Qwen2.5-3B-Instruct without service-manual context. This establishes a baseline for how much the model can answer from general knowledge alone before receiving Toyota/Lexus-specific documentation.

The evaluation includes three custom RAG-style benchmark tasks and a held-out Toyota/Lexus testing split:

1. **Held-out Toyota/Lexus testing split:** owner-style questions based on the project task.
2. **Diagnostic code benchmark:** questions about OBD-II codes and likely first checks.
3. **Service specification benchmark:** questions about fluids and specifications where source grounding matters.
4. **Noisy/safety benchmark:** questions where unsupported or overconfident repair advice would be risky.

Because this is an open-ended generation task, I calculate expected-keyword coverage, a format score for the required response structure, and an unsupported-information flag for claims marked as risky for a prompt. These measures support consistent comparison, while the sample responses are also reviewed qualitatively because an automotive answer can sound plausible while still relying on weak or incomplete support.

In [1]:
import os
import re
import html
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_DIR = Path.cwd()

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CHUNKS_PATH = PROJECT_DIR / "toyota_lexus_rag_chunks.csv"
EMBEDDINGS_PATH = PROJECT_DIR / "toyota_lexus_embeddings.npy"

GENERATION_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 250

print("Generation model:", GENERATION_MODEL_NAME)

Project directory: .
Results directory: .\results
Generation model: Qwen/Qwen2.5-3B-Instruct


In [2]:
benchmark_cases = [
    {
        "case_id": "heldout_001",
        "benchmark": "heldout_toyota_lexus_split",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My 2001 Toyota Tundra has an EVAP code P0440. What should I check first?",
        "expected_terms": ["evap", "fuel cap", "hose", "purge", "vent"],
        "must_not_terms": ["replace the engine", "ignore it"],
        "desired_response": "P0440 is related to the evaporative emissions system. First checks should include the fuel cap, EVAP hoses, purge or vent components, and any related EVAP codes. If there is a fuel smell, the vehicle should be inspected carefully for leaks rather than ignored."
    },
    {
        "case_id": "heldout_002",
        "benchmark": "heldout_toyota_lexus_split",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "My 2001 Lexus LS430 has code P0420 and drives normally. Should I replace the catalytic converter right away?",
        "expected_terms": ["p0420", "catalyst", "exhaust", "oxygen sensor", "related codes"],
        "must_not_terms": ["replace the catalytic converter immediately"],
        "desired_response": "P0420 is related to catalyst system efficiency, but the catalytic converter should not be assumed to be the only cause right away. First checks should include related codes, exhaust leaks, oxygen sensor operation, and general engine performance issues before approving an expensive catalytic converter repair."
    },
    {
        "case_id": "heldout_003",
        "benchmark": "heldout_toyota_lexus_split",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My 2001 Toyota Tundra is overheating in traffic. What should I do first?",
        "expected_terms": ["overheating", "coolant", "cool", "radiator", "fan"],
        "must_not_terms": ["keep driving", "safe to drive indefinitely"],
        "desired_response": "Overheating should be treated as serious because continuing to drive while hot can damage the engine. The safest first step is to stop safely and let the engine cool before checking coolant level. A full answer should use cooling-system manual context such as coolant level, radiator, cooling fan, thermostat, hose leak, or water pump guidance."
    },
    {
        "case_id": "dtc_001",
        "benchmark": "diagnostic_code_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My 2001 Toyota Tundra has code P0300 and is running rough. Is it safe to keep driving?",
        "expected_terms": ["p0300", "misfire", "rough", "ignition", "fuel"],
        "must_not_terms": ["drive normally", "ignore"],
        "desired_response": "P0300 means the engine is detecting random or multiple misfires. If the truck is running rough or the check engine light is flashing, it should not be driven hard because severe misfires can damage the catalytic converter. First checks should include ignition components, fuel delivery, vacuum leaks, compression concerns, and whether more specific misfire codes are present."
    },
    {
        "case_id": "dtc_002",
        "benchmark": "diagnostic_code_benchmark",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "My 2001 Lexus LS430 has code P0171. What does system too lean usually mean?",
        "expected_terms": ["lean", "air", "fuel", "vacuum", "maf"],
        "must_not_terms": ["replace the engine", "ignore"],
        "desired_response": "A lean code means the engine computer is seeing more air or less fuel than expected. First checks should include intake or vacuum leaks, mass airflow sensor issues, fuel delivery problems, exhaust leaks affecting sensor readings, and related codes."
    },
    {
        "case_id": "dtc_003",
        "benchmark": "diagnostic_code_benchmark",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "My Lexus LS430 has an EVAP leak code. What parts of the system should I think about first?",
        "expected_terms": ["evap", "fuel cap", "hose", "leak", "purge"],
        "must_not_terms": ["transmission", "brake fluid"],
        "desired_response": "An EVAP leak code points toward the evaporative emissions system. The first checks should include the fuel cap, EVAP hoses, purge and vent components, leak detection components if applicable, and any related EVAP codes."
    },
    {
        "case_id": "spec_001",
        "benchmark": "service_specification_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "What automatic transmission fluid does a 2001 Toyota Tundra 4.7L use for a drain and refill?",
        "expected_terms": ["automatic transmission", "fluid", "dexron", "drain", "refill"],
        "must_not_terms": ["engine oil", "brake fluid"],
        "desired_response": "For the 2001 Toyota Tundra 4.7L, the service information lists automatic transmission fluid as DEXRON II or DEXRON III. The owner should still verify the exact application and avoid confusing drain-and-refill capacity with dry-fill capacity."
    },
    {
        "case_id": "spec_002",
        "benchmark": "service_specification_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "What brake fluid should I use in a 2001 Toyota Tundra?",
        "expected_terms": ["brake", "fluid", "dot 3"],
        "must_not_terms": ["dexron", "engine oil"],
        "desired_response": "The service information lists DOT 3 brake fluid. Because brakes are safety-critical, the owner should use the correct fluid type and avoid mixing incompatible fluids."
    },
    {
        "case_id": "safety_001",
        "benchmark": "noisy_context_safety_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My check engine light is on but the truck drives fine. Can I just clear the code and ignore it?",
        "expected_terms": ["code", "diagnose", "related", "safety", "mechanic"],
        "must_not_terms": ["ignore it", "always safe"],
        "desired_response": "A check engine light should not automatically be ignored even if the truck drives normally. The first step is to read the diagnostic code, check for related codes, and use the correct service-manual context before deciding how urgent the issue is."
    },
    {
        "case_id": "safety_002",
        "benchmark": "noisy_context_safety_benchmark",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "If a forum post says P0420 always means the catalytic converter is bad, should I trust that?",
        "expected_terms": ["forum", "not always", "manual", "exhaust", "oxygen sensor"],
        "must_not_terms": ["always trust", "replace immediately"],
        "desired_response": "A forum post should not be treated as definitive technical evidence. P0420 can involve catalyst efficiency, but the assistant should prioritize service-manual context and check related codes, exhaust leaks, oxygen sensor operation, and engine performance before assuming the catalytic converter has failed."
    }
]

benchmark_df = pd.DataFrame(benchmark_cases)
benchmark_df

,case_id,benchmark,vehicle_make,vehicle_model,question,expected_terms,must_not_terms,desired_response
0,heldout_001,heldout_toyota_lexus_split,Toyota,Tundra,My 2001 Toyota Tundra has an EVAP code P0440. ...,"[evap, fuel cap, hose, purge, vent]","[replace the engine, ignore it]",P0440 is related to the evaporative emissions ...
1,heldout_002,heldout_toyota_lexus_split,Lexus,LS430,My 2001 Lexus LS430 has code P0420 and drives ...,"[p0420, catalyst, exhaust, oxygen sensor, rela...",[replace the catalytic converter immediately],P0420 is related to catalyst system efficiency...
2,heldout_003,heldout_toyota_lexus_split,Toyota,Tundra,My 2001 Toyota Tundra is overheating in traffi...,"[overheating, coolant, cool, radiator, fan]","[keep driving, safe to drive indefinitely]",Overheating should be treated as serious becau...
3,dtc_001,diagnostic_code_benchmark,Toyota,Tundra,My 2001 Toyota Tundra has code P0300 and is ru...,"[p0300, misfire, rough, ignition, fuel]","[drive normally, ignore]",P0300 means the engine is detecting random or ...
4,dtc_002,diagnostic_code_benchmark,Lexus,LS430,My 2001 Lexus LS430 has code P0171. What does ...,"[lean, air, fuel, vacuum, maf]","[replace the engine, ignore]",A lean code means the engine computer is seein...
5,dtc_003,diagnostic_code_benchmark,Lexus,LS430,My Lexus LS430 has an EVAP leak code. What par...,"[evap, fuel cap, hose, leak, purge]","[transmission, brake fluid]",An EVAP leak code points toward the evaporativ...
6,spec_001,service_specification_benchmark,Toyota,Tundra,What automatic transmission fluid does a 2001 ...,"[automatic transmission, fluid, dexron, drain,...","[engine oil, brake fluid]","For the 2001 Toyota Tundra 4.7L, the service i..."
7,spec_002,service_specification_benchmark,Toyota,Tundra,What brake fluid should I use in a 2001 Toyota...,"[brake, fluid, dot 3]","[dexron, engine oil]",The service information lists DOT 3 brake flui...
8,safety_001,noisy_context_safety_benchmark,Toyota,Tundra,My check engine light is on but the truck driv...,"[code, diagnose, related, safety, mechanic]","[ignore it, always safe]",A check engine light should not automatically ...
9,safety_002,noisy_context_safety_benchmark,Lexus,LS430,If a forum post says P0420 always means the ca...,"[forum, not always, manual, exhaust, oxygen se...","[always trust, replace immediately]",A forum post should not be treated as definiti...


In [3]:
def normalize_text(text):
    return str(text).lower()


def keyword_coverage(response, expected_terms):
    response_lower = normalize_text(response)
    matches = [term for term in expected_terms if term.lower() in response_lower]
    return len(matches) / max(len(expected_terms), 1), matches


def format_score(response):
    response_lower = normalize_text(response)
    headings = [
        "what this likely means",
        "likely causes",
        "first checks",
        "safety",
        "mechanic"
    ]
    return sum(h in response_lower for h in headings) / len(headings)


def unsupported_flag(response, must_not_terms):
    response_lower = normalize_text(response)
    bad_matches = [term for term in must_not_terms if term.lower() in response_lower]
    return int(len(bad_matches) > 0), bad_matches


def score_response(row, response):
    coverage, matched_terms = keyword_coverage(response, row["expected_terms"])
    unsupported, unsupported_terms = unsupported_flag(response, row["must_not_terms"])
    structure = format_score(response)

    overall_score = (0.55 * coverage) + (0.30 * structure) + (0.15 * (1 - unsupported))

    return {
        "keyword_coverage": coverage,
        "matched_terms": matched_terms,
        "format_score": structure,
        "unsupported_flag": unsupported,
        "unsupported_terms": unsupported_terms,
        "overall_score": overall_score
    }


def print_sample_responses(results_df, n=2):
    sample_df = results_df.sample(min(n, len(results_df)), random_state=42)

    for _, row in sample_df.iterrows():
        print("=" * 100)
        print("Benchmark:", row["benchmark"])
        print("Question:")
        print(row["question"])
        print("\nModel Response:")
        print(row["model_response"])
        print("\nDesired Response:")
        print(row["desired_response"])
        print("=" * 100)

In [4]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import logging

logging.set_verbosity_error()
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def load_generation_model(model_name=GENERATION_MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        clean_up_tokenization_spaces=False
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Enable GPU acceleration with 4-bit quantization when CUDA is available.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print("Using GPU:", torch.cuda.get_device_name(0))

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=quantization_config,
            low_cpu_mem_usage=True,
            attn_implementation="eager"
        )
    else:
        print("CUDA not available. Using CPU.")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map={"": "cpu"},
            dtype=torch.float32,
            low_cpu_mem_usage=True,
            attn_implementation="eager"
        )

    model.eval()
    print("Model loaded:", model_name)

    if hasattr(model, "hf_device_map"):
        print("Device map:", model.hf_device_map)

    return tokenizer, model

def get_model_context_limit(model, tokenizer):
    limits = []

    if hasattr(model.config, "max_position_embeddings") and model.config.max_position_embeddings:
        limits.append(model.config.max_position_embeddings)

    if hasattr(tokenizer, "model_max_length") and tokenizer.model_max_length:
        if tokenizer.model_max_length < 100000:
            limits.append(tokenizer.model_max_length)

    return min(limits) if limits else 2048


def format_chat_prompt(tokenizer, prompt):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful Toyota/Lexus repair explanation assistant. "
                "Do not invent unsupported repair details. If source context is missing, say what information is still needed."
            )
        },
        {"role": "user", "content": prompt}
    ]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        return messages[0]["content"] + "\n\n" + prompt


def generate_model_response(tokenizer, model, prompt, max_new_tokens=MAX_NEW_TOKENS):
    formatted_prompt = format_chat_prompt(tokenizer, prompt)

    context_limit = get_model_context_limit(model, tokenizer)
    input_budget = max(256, context_limit - max_new_tokens)

    old_truncation_side = tokenizer.truncation_side
    tokenizer.truncation_side = "left"

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=input_budget
    )

    tokenizer.truncation_side = old_truncation_side

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )


def build_no_rag_prompt(question):
    return f"""
Answer the Toyota/Lexus owner question below in plain English.

Owner Question:
{question}

No retrieved service-manual context is available for this baseline test.

Write the answer in this format:
1. What this likely means
2. Likely causes or relevant specs
3. First checks
4. Safety concerns
5. When to see a mechanic

Important rules:
- Do not invent torque specs, part locations, repair steps, or safety claims.
- If source context is needed, say what information is still needed.
""".strip()

In [5]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("CUDA available:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0))
    print("Allocated before loading:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
    print("Reserved before loading:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti
Allocated before loading: 0.0 GB
Reserved before loading: 0.0 GB


In [6]:
tokenizer, model = load_generation_model(GENERATION_MODEL_NAME)

pre_rag_rows = []

for _, row in benchmark_df.iterrows():
    prompt = build_no_rag_prompt(row["question"])
    response = generate_model_response(tokenizer, model, prompt)

    score_info = score_response(row, response)

    pre_rag_rows.append({
        "case_id": row["case_id"],
        "benchmark": row["benchmark"],
        "question": row["question"],
        "model_response": response,
        "desired_response": row["desired_response"],
        **score_info
    })

pre_rag_results = pd.DataFrame(pre_rag_rows)
pre_rag_results.to_csv(RESULTS_DIR / "pre_rag_eval_results.csv", index=False)

pre_rag_summary = pre_rag_results.groupby("benchmark")[[
    "keyword_coverage",
    "format_score",
    "unsupported_flag",
    "overall_score"
]].mean().reset_index()

pre_rag_summary

Using GPU: NVIDIA GeForce RTX 4070 Ti


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-3B-Instruct


,benchmark,keyword_coverage,format_score,unsupported_flag,overall_score
0,diagnostic_code_benchmark,0.733333,0.800000,0.0,0.793333
1,heldout_toyota_lexus_split,0.733333,0.933333,0.0,0.833333
2,noisy_context_safety_benchmark,0.400000,0.700000,0.0,0.580000
3,service_specification_benchmark,1.000000,1.000000,0.0,1.000000


In [7]:
print_sample_responses(pre_rag_results, n=2)

Benchmark: noisy_context_safety_benchmark
Question:
My check engine light is on but the truck drives fine. Can I just clear the code and ignore it?

Model Response:
1. **What this likely means:**
   The check engine light coming on could mean there's an issue with your vehicle that needs attention, even if it doesn't affect how the truck drives right now. Ignoring it might lead to more serious problems down the line.

2. **Likely causes or relevant specs:**
   There are various reasons why the check engine light might come on, such as a malfunctioning sensor, a dirty oxygen sensor, a faulty catalytic converter, or a problem with the emission control system. These issues can be related to specific parts of the engine or emission control systems.

3. **First checks:**
   - Check if the gas cap is tight. A loose gas cap can cause the check engine light to turn on temporarily.
   - Look for any obvious signs of damage or leaks, like oil or coolant seeping from the engine.
   - Check the ai

## 3. Service-Manual RAG Pipeline

The RAG pipeline is built from the Toyota and Lexus service-manual folders. The parser loads the HTML pages, extracts their text, preserves metadata such as vehicle make, model, year, and page title, divides the pages into chunks, creates retrieval representations, and returns the most relevant passages for each question.

A small set of forum-style owner-language examples was also tested in a separate run. That version made the dataset more similar to the way owners ask questions, but final benchmark performance was worse than the service-manual-only implementation. The final pipeline therefore remains focused on service-manual context, while the commented experiment is retained below to document the comparison.

Three retrieval combinations are evaluated:

1. **TF-IDF + cosine similarity** as a keyword-focused baseline.
2. **Sentence embeddings + cosine similarity** for semantic retrieval.
3. **Sentence embeddings + Euclidean distance** as a second embedding-distance comparison.

The methods are compared on the same manually constructed prompts, and the strongest setup is selected for the final pipeline.

In [8]:
from bs4 import BeautifulSoup

MANUAL_SOURCES = [
    {
        "folder": PROJECT_DIR / "2001 Toyota Tundra Limited, 4WD",
        "source_name": "2001 Toyota Tundra Service Manual",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "model_year": "2001",
        "trim": "Limited, 4WD"
    },
    {
        "folder": PROJECT_DIR / "2001 Lexus LS 430",
        "source_name": "2001 Lexus LS430 Service Manual",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "model_year": "2001",
        "trim": "Base"
    }
]

MAX_HTML_FILES_PER_SOURCE = 5000

for source in MANUAL_SOURCES:
    print(source["source_name"])
    print("Exists:", source["folder"].exists())
    print()

2001 Toyota Tundra Service Manual
Folder: .\2001 Toyota Tundra Limited, 4WD
Exists: True

2001 Lexus LS430 Service Manual
Folder: .\2001 Lexus LS 430
Exists: True



In [9]:
def clean_text(text):
    text = html.unescape(str(text)) if text is not None else ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def table_to_text(table):
    rows = []
    for tr in table.find_all("tr"):
        cells = [clean_text(c.get_text(" ", strip=True)) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        if cells:
            rows.append(" | ".join(cells))
    return "\n".join(rows)


def get_breadcrumbs(soup):
    parts = [clean_text(a.get_text(" ", strip=True)) for a in soup.select(".breadcrumb-part")]
    parts = [p for p in parts if p and p.lower() != "home"]
    return " > ".join(parts)


def classify_section(title, breadcrumb, text):
    combined = f"{title} {breadcrumb} {text[:1000]}".lower()

    if "dtc" in combined or "diagnostic trouble code" in combined:
        return "dtc_diagnostics"
    if "technical service bulletin" in combined or "tsb" in combined:
        return "technical_service_bulletin"
    if "fluid" in combined:
        return "fluids"
    if "specification" in combined or "torque" in combined:
        return "specifications"
    if "wiring" in combined or "electrical diagram" in combined:
        return "wiring_diagram"
    if "component location" in combined or "locations" in combined:
        return "component_location"
    if "labor" in combined:
        return "labor_times"
    if "repair" in combined or "procedure" in combined or "remove" in combined or "install" in combined:
        return "repair_procedure"
    return "manual_page"


def should_skip_page(file_path, h1):
    file_name = str(file_path).lower()
    h1_lower = str(h1).lower()

    if not file_name.endswith(".html"):
        return True
    if file_name.endswith("404.html") or file_name.endswith("about.html") or file_name.endswith("external-car.html"):
        return True

    generic_other_makers = ["chrysler group llc", "ford motor co.", "general motors"]

    if h1_lower in generic_other_makers:
        return True

    return False


def chunk_text(text, max_chars=2500, overlap=350):
    text = clean_text(text)

    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars

        if end < len(text):
            boundary = max(
                text.rfind(". ", start, end),
                text.rfind("; ", start, end),
                text.rfind("\n", start, end)
            )

            if boundary > start + int(max_chars * 0.60):
                end = boundary + 1

        chunks.append(text[start:end].strip())

        if end >= len(text):
            break

        start = max(0, end - overlap)

    return chunks

In [10]:
def parse_manual_page(file_path, source_info):
    raw_html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw_html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    title = clean_text(soup.title.get_text(" ", strip=True)) if soup.title else ""
    breadcrumb = get_breadcrumbs(soup)

    main = soup.find("div", class_="main") or soup.body
    if main is None:
        return None

    h1_tag = main.find("h1")
    h1 = clean_text(h1_tag.get_text(" ", strip=True)) if h1_tag else title

    if should_skip_page(file_path, h1):
        return None

    warnings_list = [clean_text(w.get_text(" ", strip=True)) for w in main.select(".other-warning, .WARNING")]
    notes = [clean_text(n.get_text(" ", strip=True)) for n in main.select(".NOTE")]

    tables = [table_to_text(t) for t in main.find_all("table")]
    tables = [t for t in tables if t]

    image_captions = []
    for cap in main.select(".imageCaption"):
        image_captions.append(clean_text(cap.get_text(" ", strip=True)))

    for img in main.find_all("img"):
        alt = clean_text(img.get("alt", ""))
        src = clean_text(img.get("src", ""))
        if alt or src:
            image_captions.append(f"Image: {alt} ({src})" if alt else f"Image: {src}")

    image_captions = list(dict.fromkeys([c for c in image_captions if c]))

    body_text = clean_text(main.get_text(" ", strip=True))
    tables_text = "\n\n".join(tables)
    image_text = " | ".join(image_captions)

    combined_parts = [body_text]

    if tables_text and tables_text not in body_text:
        combined_parts.append("Tables:\n" + tables_text)

    if image_text:
        combined_parts.append("Image captions/files:\n" + image_text)

    combined_text = "\n\n".join([p for p in combined_parts if p])

    row = {
        "source_name": source_info["source_name"],
        "source_type": "service_manual",
        "vehicle_make": source_info["vehicle_make"],
        "vehicle_model": source_info["vehicle_model"],
        "model_year": source_info["model_year"],
        "trim": source_info["trim"],
        "source_file": str(file_path),
        "page_title": h1,
        "html_title": title,
        "breadcrumb_path": breadcrumb,
        "section_type": classify_section(title, breadcrumb, body_text),
        "warning_text": " ".join(warnings_list),
        "note_text": " ".join(notes[:5]),
        "image_captions": image_text,
        "combined_text": combined_text
    }

    return row


def build_chunks_from_sources(sources, max_html_files_per_source=None):
    rows = []

    for source_info in sources:
        folder_path = source_info["folder"]

        if not folder_path.exists():
            print(f"Skipping missing folder: {folder_path}")
            continue

        html_files = list(folder_path.rglob("*.html"))

        if max_html_files_per_source is not None:
            html_files = html_files[:max_html_files_per_source]

        print(f"{source_info['source_name']}: {len(html_files)} HTML files to inspect")

        for file_path in html_files:
            try:
                page = parse_manual_page(file_path, source_info)
                if page is None:
                    continue

                chunks = chunk_text(page["combined_text"], max_chars=2500, overlap=350)

                for chunk_index, chunk in enumerate(chunks):
                    if len(chunk) < 80:
                        continue

                    row = {k: v for k, v in page.items() if k != "combined_text"}
                    row["chunk_index"] = chunk_index
                    row["chunk_id"] = f"{source_info['vehicle_make']}_{source_info['vehicle_model']}_{file_path.stem}_{chunk_index}"
                    row["text_chunk"] = chunk
                    rows.append(row)

            except Exception:
                continue

    return pd.DataFrame(rows)

In [11]:
chunks_df = build_chunks_from_sources(
    MANUAL_SOURCES,
    max_html_files_per_source=MAX_HTML_FILES_PER_SOURCE
)

chunks_df.to_csv(CHUNKS_PATH, index=False)

print("Rows:", len(chunks_df))

chunks_df.drop(columns=["source_file"], errors="ignore").head()

2001 Toyota Tundra Service Manual: 5000 HTML files to inspect
2001 Lexus LS430 Service Manual: 5000 HTML files to inspect
Saved: .\toyota_lexus_rag_chunks.csv
Rows: 15418


source_name     source_type vehicle_make  \
0  2001 Toyota Tundra Service Manual  service_manual       Toyota   
1  2001 Toyota Tundra Service Manual  service_manual       Toyota   
2  2001 Toyota Tundra Service Manual  service_manual       Toyota   
3  2001 Toyota Tundra Service Manual  service_manual       Toyota   
4  2001 Toyota Tundra Service Manual  service_manual       Toyota   

  vehicle_model model_year          trim  \
0        Tundra       2001  Limited, 4WD   
1        Tundra       2001  Limited, 4WD   
2        Tundra       2001  Limited, 4WD   
3        Tundra       2001  Limited, 4WD   
4        Tundra       2001  Limited, 4WD   

                                         source_file  \
0  [local path removed]
1  [local path removed]
2  [local path removed]
3  [local path removed]
4  [local path removed]

                             page_title  \
0     Toyota: 2001: Tundra Limited, 4WD   
1     Toyota: 2001: Tundra Limited, 4WD   
2    Door Lock Switch: Remove & Replace   
3  Load Sensing Valve: Remove & Replace   
4    Initial (Factory) Feature Settings   

                                          html_title  \
0  Free Service Manual for the 2001 Toyota Tundra...   
1  Free Service Manual for the 2001 Toyota Tundra...   
2  Door Lock Switch: Remove & Replace — 2001 Toyo...   
3  Load Sensing Valve: Remove & Replace — 2001 To...   
4  Initial (Factory) Feature Settings — 2001 Toyo...   

                                     breadcrumb_path      section_type  \
0                Toyota > 2001 > Tundra Limited, 4WD            fluids   
1                Toyota > 2001 > Tundra Limited, 4WD            fluids   
2  Toyota > 2001 > Tundra Limited, 4WD > Labor Ti...       labor_times   
3  Toyota > 2001 > Tundra Limited, 4WD > Labor Ti...       labor_times   
4  Toyota > 2001 > Tundra Limited, 4WD > Repair a...  repair_procedure   

                                        warning_text note_text  \
0                                                                
1                                                                
2  WARNING: This page is about the Tundra Base, A...             
3  WARNING: This page is about the Tundra Base, A...             
4                                                                

                                      image_captions  chunk_index  \
0  Image: icons/wrench.svg | Image: icons/tools-a...            0   
1  Image: ../icons/wrench.svg | Image: ../icons/t...            0   
2                                                               0   
3                                                               0   
4                                                               0   

                chunk_id                                         text_chunk  
0  Toyota_Tundra_index_0  Toyota: 2001: Tundra Limited, 4WD This is a LE...  
1      Toyota_Tundra_1_0  Toyota: 2001: Tundra Limited, 4WD This is a LE...  
2     Toyota_Tundra_10_0  Door Lock Switch: Remove & Replace WARNING: Th...  
3    Toyota_Tundra_100_0  Load Sensing Valve: Remove & Replace WARNING: ...  
4   Toyota_Tundra_1000_0  Initial (Factory) Feature Settings FEATURE SET...

In [12]:
chunks_df.groupby(["vehicle_make", "vehicle_model", "section_type"]).size().reset_index(name="count").sort_values(
    ["vehicle_make", "count"], ascending=[True, False]
).head(20)

,vehicle_make,vehicle_model,section_type,count
4,Lexus,LS430,repair_procedure,2929
6,Lexus,LS430,technical_service_bulletin,1233
1,Lexus,LS430,dtc_diagnostics,1189
7,Lexus,LS430,wiring_diagram,904
5,Lexus,LS430,specifications,432
0,Lexus,LS430,component_location,196
2,Lexus,LS430,fluids,183
3,Lexus,LS430,labor_times,104
12,Toyota,Tundra,repair_procedure,3377
14,Toyota,Tundra,technical_service_bulletin,1646


### Forum-Style Owner-Language Experiment

Curated forum-style examples were tested to help the retriever connect normal owner wording with technical terminology. The service-manual-only version produced stronger final benchmark results, so the forum rows are not included in the final corpus. The commented code below documents the experiment without changing the final chunks, embeddings, retrieval results, or benchmark scores.

In [13]:
# FORUM DATA EXPERIMENT - NOT USED IN FINAL PIPELINE
#
# I tested this owner-language forum data in a separate run, but the model performed worse
# after including the forum rows. Because the final system should stay grounded in
# service-manual context, this code is intentionally commented out and does not run.
#
# FORUM_DATA_PATH = PROJECT_DIR / "forum_examples.csv"
#
# if not FORUM_DATA_PATH.exists():
#     raise FileNotFoundError(
#         "forum_examples.csv was not found. This file was used only for an experimental forum-data run."
#     )
#
# forum_df = pd.read_csv(FORUM_DATA_PATH)
#
# required_columns = ["vehicle_make", "vehicle_model", "model_year", "question", "accepted_answer", "url"]
# missing_columns = [col for col in required_columns if col not in forum_df.columns]
#
# if missing_columns:
#     raise ValueError(f"forum_examples.csv is missing required columns: {missing_columns}")
#
# if "trim" not in forum_df.columns:
#     forum_df["trim"] = ""
#
# forum_rows = []
#
# for i, row in forum_df.iterrows():
#     text = f"Owner question: {row['question']}\nCurated forum summary: {row['accepted_answer']}"
#
#     forum_rows.append({
#         "source_name": "Curated owner forum examples",
#         "source_type": "forum_owner_language",
#         "vehicle_make": row["vehicle_make"],
#         "vehicle_model": row["vehicle_model"],
#         "model_year": row["model_year"],
#         "trim": row.get("trim", ""),
#         "source_file": str(FORUM_DATA_PATH),
#         "page_title": row["question"],
#         "html_title": "",
#         "breadcrumb_path": "Forum examples",
#         "section_type": "forum_owner_language",
#         "warning_text": "Forum content is owner-generated and was tested only as owner-language retrieval support.",
#         "note_text": "",
#         "image_captions": "",
#         "chunk_index": 0,
#         "chunk_id": f"forum_{i}",
#         "text_chunk": text,
#         "url": row["url"]
#     })
#
# forum_chunks_df = pd.DataFrame(forum_rows)
#
# # This line is intentionally commented out because forum rows are not part of the final RAG dataset.
# # chunks_df = pd.concat([chunks_df, forum_chunks_df], ignore_index=True)
# # chunks_df.to_csv(CHUNKS_PATH, index=False)
#
# # print("Forum owner-language rows added:", len(forum_chunks_df))
# # print("Updated RAG chunk count:", len(chunks_df))
# # chunks_df.groupby("source_type").size().reset_index(name="count")

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)
tfidf_matrix = tfidf_vectorizer.fit_transform(chunks_df["text_chunk"].fillna(""))

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (15418, 17286)


In [15]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError(
        "sentence-transformers is required for the embedding retrieval section. "
        "Install it with: pip install sentence-transformers"
    ) from exc

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=EMBEDDING_DEVICE)

texts_for_embedding = chunks_df["text_chunk"].fillna("").tolist()

embeddings = embedding_model.encode(
    texts_for_embedding,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

np.save(EMBEDDINGS_PATH, embeddings)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding device:", EMBEDDING_DEVICE)
print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/241 [00:00<?, ?it/s]

Embedding model: BAAI/bge-small-en-v1.5
Embedding device: cuda
Embeddings shape: (15418, 384)
Saved embeddings: .\toyota_lexus_embeddings.npy


In [16]:
cosine_nn = NearestNeighbors(n_neighbors=10, metric="cosine")
cosine_nn.fit(embeddings)

euclidean_nn = NearestNeighbors(n_neighbors=10, metric="euclidean")
euclidean_nn.fit(embeddings)

print("NearestNeighbors indices created.")

NearestNeighbors indices created.


In [17]:
def infer_vehicle_filter(question):
    question_lower = question.lower()

    if "lexus" in question_lower or "ls430" in question_lower or "ls 430" in question_lower:
        return {"vehicle_make": "Lexus"}

    if "toyota" in question_lower or "tundra" in question_lower:
        return {"vehicle_make": "Toyota"}

    return {}


def apply_vehicle_filter(df, question):
    filters = infer_vehicle_filter(question)
    filtered = df.copy()

    for col, value in filters.items():
        filtered = filtered[filtered[col].astype(str).str.lower() == str(value).lower()]

    if len(filtered) == 0:
        return df.copy()

    return filtered


def retrieve_tfidf_cosine(query, k=3):
    search_df = apply_vehicle_filter(chunks_df, query)
    matrix = tfidf_vectorizer.transform(search_df["text_chunk"].fillna(""))

    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, matrix).flatten()
    top_idx = scores.argsort()[::-1][:k]

    results = search_df.iloc[top_idx].copy()
    results["similarity_score"] = scores[top_idx]
    return results


def retrieve_embedding_cosine(query, k=3):
    search_df = apply_vehicle_filter(chunks_df, query)
    search_indices = search_df.index.to_numpy()

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sub_embeddings = embeddings[search_indices]
    nn = NearestNeighbors(n_neighbors=min(k, len(search_df)), metric="cosine")
    nn.fit(sub_embeddings)

    distances, local_indices = nn.kneighbors(query_embedding, n_neighbors=min(k, len(search_df)))

    results = search_df.iloc[local_indices[0]].copy()
    results["similarity_score"] = 1 - distances[0]
    return results


def retrieve_embedding_euclidean(query, k=3):
    search_df = apply_vehicle_filter(chunks_df, query)
    search_indices = search_df.index.to_numpy()

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sub_embeddings = embeddings[search_indices]
    nn = NearestNeighbors(n_neighbors=min(k, len(search_df)), metric="euclidean")
    nn.fit(sub_embeddings)

    distances, local_indices = nn.kneighbors(query_embedding, n_neighbors=min(k, len(search_df)))

    results = search_df.iloc[local_indices[0]].copy()
    results["similarity_score"] = -distances[0]
    return results


retrievers = {
    "tfidf_cosine": retrieve_tfidf_cosine,
    "embedding_cosine": retrieve_embedding_cosine,
    "embedding_euclidean": retrieve_embedding_euclidean
}

In [18]:
def score_retrieval(question, expected_terms, retrieved_df):
    combined_text = " ".join(
        retrieved_df["page_title"].fillna("").astype(str).tolist()
        + retrieved_df["breadcrumb_path"].fillna("").astype(str).tolist()
        + retrieved_df["text_chunk"].fillna("").astype(str).tolist()
    ).lower()

    matched_terms = [term for term in expected_terms if term.lower() in combined_text]
    score = len(matched_terms) / max(len(expected_terms), 1)

    return score, matched_terms


retrieval_eval_rows = []

for retriever_name, retriever_fn in retrievers.items():
    for _, row in benchmark_df.iterrows():
        retrieved = retriever_fn(row["question"], k=3)
        score, matched_terms = score_retrieval(row["question"], row["expected_terms"], retrieved)

        retrieval_eval_rows.append({
            "retriever": retriever_name,
            "case_id": row["case_id"],
            "benchmark": row["benchmark"],
            "question": row["question"],
            "retrieval_score": score,
            "matched_terms": matched_terms,
            "top_titles": " | ".join(retrieved["page_title"].astype(str).head(3))
        })

retrieval_eval_df = pd.DataFrame(retrieval_eval_rows)
retrieval_eval_df.to_csv(RESULTS_DIR / "retrieval_method_comparison.csv", index=False)

retrieval_summary = retrieval_eval_df.groupby("retriever")["retrieval_score"].mean().reset_index().sort_values(
    "retrieval_score",
    ascending=False
)

retrieval_summary

,retriever,retrieval_score
2,tfidf_cosine,0.540000
0,embedding_cosine,0.246667
1,embedding_euclidean,0.246667


In [19]:
retrieval_eval_df[["retriever", "case_id", "retrieval_score", "matched_terms", "top_titles"]].head(12)

,retriever,case_id,retrieval_score,matched_terms,top_titles
0,tfidf_cosine,heldout_001,0.6,"[evap, purge, vent]","Emission Control Abbreviations: ""E"" | Emission..."
1,tfidf_cosine,heldout_002,0.8,"[p0420, catalyst, exhaust, oxygen sensor]",Circuit Description | Oxidation Catalyst (OC) ...
2,tfidf_cosine,heldout_003,0.0,[],EVAP System Operation Information (EG005-01) |...
3,tfidf_cosine,dtc_001,0.4,"[p0300, misfire]",Diagnosis & Repair | Diagnosis & Repair | Diag...
4,tfidf_cosine,dtc_002,0.4,"[lean, air]",DTC P0171/P0172: System Too Lean/Rich (Bank No...
5,tfidf_cosine,dtc_003,0.6,"[evap, leak, purge]","Emission Control Abbreviations: ""E"" | Emission..."
6,tfidf_cosine,spec_001,1.0,"[automatic transmission, fluid, dexron, drain,...",Quick Lookups: Fluids | Quick Lookups: Fluids ...
7,tfidf_cosine,spec_002,1.0,"[brake, fluid, dot 3]",Brake Systems: Fluid | Brake Fluid | Brake Sys...
8,tfidf_cosine,safety_001,0.2,[code],Erasing Diagnostic Trouble Codes | Erasing Dia...
9,tfidf_cosine,safety_002,0.4,"[exhaust, oxygen sensor]",Circuit Description | Oxidation Catalyst (OC) ...


### Retrieval Method Choice

TF-IDF with cosine similarity performed best on the service-manual-only dataset, with an average retrieval score of 0.54. Both embedding-based methods scored approximately 0.25. Although sentence embeddings often improve semantic matching, the result is reasonable for this project because many automotive questions depend on exact diagnostic codes, fluid names, and system abbreviations.

The final pipeline therefore uses TF-IDF with cosine similarity. A future version could combine exact DTC and keyword matching with embeddings so the system can better handle both technical codes and broader owner-language descriptions.

In [20]:
BEST_RETRIEVER_NAME = retrieval_summary.iloc[0]["retriever"]
best_retriever = retrievers[BEST_RETRIEVER_NAME]

print("Best retriever selected:", BEST_RETRIEVER_NAME)

example_retrieval = best_retriever("2001 Lexus LS430 P0420 catalyst efficiency oxygen sensor exhaust leak", k=3)
example_retrieval[["similarity_score", "vehicle_make", "vehicle_model", "page_title", "section_type", "breadcrumb_path"]]

Best retriever selected: tfidf_cosine


,similarity_score,vehicle_make,vehicle_model,page_title,section_type,breadcrumb_path
12226,0.274765,Lexus,LS430,DTC P0420/P0430: Catalyst System Efficiency Be...,dtc_diagnostics,Lexus > 2001 > LS 430 > Repair and Diagnosis >...
12229,0.254854,Lexus,LS430,Diagnosis & Repair,dtc_diagnostics,Lexus > 2001 > LS 430 > Repair and Diagnosis >...
12227,0.214117,Lexus,LS430,Circuit Description,dtc_diagnostics,Lexus > 2001 > LS 430 > Repair and Diagnosis >...


In [21]:
class ToyotaLexusRAGPipeline:
    def __init__(self, retriever_fn, tokenizer, model, top_k=3):
        self.retriever_fn = retriever_fn
        self.tokenizer = tokenizer
        self.model = model
        self.top_k = top_k

    def retrieve(self, question):
        return self.retriever_fn(question, k=self.top_k)

    def build_context(self, retrieved_df, max_chunk_chars=1000):
        context_parts = []

        for i, row in retrieved_df.iterrows():
            text = clean_text(row["text_chunk"])

            if len(text) > max_chunk_chars:
                text = text[:max_chunk_chars].rsplit(" ", 1)[0] + "..."

            warning = row.get("warning_text", "")
            warning_line = f"Warning: {warning}\n" if isinstance(warning, str) and warning.strip() else ""

            context_parts.append(
                f"""
Source {len(context_parts) + 1}
Vehicle: {row['model_year']} {row['vehicle_make']} {row['vehicle_model']} {row['trim']}
Source Type: {row['source_type']}
Page Title: {row['page_title']}
Section Type: {row['section_type']}
Breadcrumb: {row['breadcrumb_path']}
{warning_line}Text:
{text}
""".strip()
            )

        return "\n\n---\n\n".join(context_parts)

    def build_prompt(self, question, retrieved_df):
        context = self.build_context(retrieved_df)

        return f"""
You are a Toyota/Lexus repair explanation assistant. Use only the retrieved service-manual context to answer the owner's question.

Owner Question:
{question}

Retrieved Context:
{context}

Write the answer in this format:
1. What this likely means
2. Likely causes or relevant specs
3. First checks
4. Safety concerns
5. When to see a mechanic

Important rules:
- Do not invent torque specs, part locations, repair steps, or safety claims.
- If the retrieved context is not enough, say what information is still needed.
- Explain the answer in plain English for a vehicle owner.
""".strip()

    def answer(self, question):
        retrieved_df = self.retrieve(question)
        prompt = self.build_prompt(question, retrieved_df)
        response = generate_model_response(self.tokenizer, self.model, prompt)

        return response, retrieved_df, prompt


rag_pipeline = ToyotaLexusRAGPipeline(
    retriever_fn=best_retriever,
    tokenizer=tokenizer,
    model=model,
    top_k=3
)

print("RAG pipeline ready.")

RAG pipeline ready.


In [22]:
test_question = "My 2001 Lexus LS430 has code P0420 and drives normally. Should I replace the catalytic converter right away?"

test_response, test_retrieved, test_prompt = rag_pipeline.answer(test_question)

print("Retrieved pages:")
display(test_retrieved[["similarity_score", "vehicle_make", "vehicle_model", "page_title", "section_type"]])

print("\nRAG response:")
print(test_response)

Retrieved pages:


,similarity_score,vehicle_make,vehicle_model,page_title,section_type
12227,0.369432,Lexus,LS430,Circuit Description,dtc_diagnostics
12571,0.270040,Lexus,LS430,Oxidation Catalyst (OC),repair_procedure
11085,0.230089,Lexus,LS430,Catalytic Converters,repair_procedure



RAG response:
1. **What this likely means**
   The code P0420 indicates that the oxygen sensor in the first bank of the catalytic converter is not switching back and forth as it should. This suggests that the catalytic converter might be failing, but it doesn't necessarily mean you need to replace it immediately. Your car is still driving normally, which is good news.

2. **Likely causes or relevant specs**
   The issue could be due to a faulty oxygen sensor, a faulty catalytic converter, or an exhaust gas leak. The catalytic converter is a crucial component that helps convert harmful emissions into less harmful ones. It contains pellets or monoliths and uses platinum and palladium (or just platinum) as catalysts.

3. **First checks**
   - Check the oxygen sensors in front and behind the catalytic converter. They should be working correctly. You can do this by checking their resistance values using a multimeter.
   - Inspect the catalytic converter itself for any signs of damage, such

## 4. Post-RAG Evaluation

The same benchmark is repeated after retrieval is added. For each question, the model receives relevant Toyota or Lexus service-manual passages before generating its response. Comparing these results with the baseline shows whether source-grounded context improves the same diagnostic, specification, safety, and held-out questions.

In [23]:
post_rag_rows = []

for _, row in benchmark_df.iterrows():
    response, retrieved_df, prompt = rag_pipeline.answer(row["question"])

    score_info = score_response(row, response)
    retrieval_score, retrieval_matches = score_retrieval(row["question"], row["expected_terms"], retrieved_df)

    post_rag_rows.append({
        "case_id": row["case_id"],
        "benchmark": row["benchmark"],
        "question": row["question"],
        "model_response": response,
        "desired_response": row["desired_response"],
        "retriever": BEST_RETRIEVER_NAME,
        "retrieval_score": retrieval_score,
        "retrieval_matched_terms": retrieval_matches,
        "retrieved_titles": " | ".join(retrieved_df["page_title"].astype(str).head(3)),
        **score_info
    })

post_rag_results = pd.DataFrame(post_rag_rows)
post_rag_results.to_csv(RESULTS_DIR / "post_rag_eval_results.csv", index=False)

post_rag_summary = post_rag_results.groupby("benchmark")[[
    "retrieval_score",
    "keyword_coverage",
    "format_score",
    "unsupported_flag",
    "overall_score"
]].mean().reset_index()

post_rag_summary

,benchmark,retrieval_score,keyword_coverage,format_score,unsupported_flag,overall_score
0,diagnostic_code_benchmark,0.466667,0.866667,0.800000,0.0,0.866667
1,heldout_toyota_lexus_split,0.466667,0.733333,0.733333,0.0,0.773333
2,noisy_context_safety_benchmark,0.300000,0.600000,0.800000,0.0,0.720000
3,service_specification_benchmark,1.000000,1.000000,1.000000,0.0,1.000000


In [24]:
print_sample_responses(post_rag_results, n=2)

Benchmark: noisy_context_safety_benchmark
Question:
My check engine light is on but the truck drives fine. Can I just clear the code and ignore it?

Model Response:
1. **What this likely means**
   - The check engine light being on doesn't necessarily mean there's a serious problem with your car. It could be a minor issue that's been resolved, or it might be a code that's no longer relevant because the problem has been fixed.

2. **Likely causes or relevant specs**
   - The check engine light can come on due to various issues like a faulty oxygen sensor, a malfunctioning catalytic converter, or even a loose gas cap. In some cases, it might be a code that's no longer relevant if the underlying issue has been resolved.

3. **First checks**
   - Before deciding to clear the code, you should first check if the issue that caused the code is still present. For example, if you recently had a tire rotation done, the code might have been triggered by the new tire positions. You should also chec

In [25]:
comparison_df = pre_rag_summary.merge(
    post_rag_summary,
    on="benchmark",
    suffixes=("_pre_rag", "_post_rag")
)

comparison_df["overall_score_change"] = comparison_df["overall_score_post_rag"] - comparison_df["overall_score_pre_rag"]
comparison_df["keyword_coverage_change"] = comparison_df["keyword_coverage_post_rag"] - comparison_df["keyword_coverage_pre_rag"]
comparison_df["unsupported_flag_change"] = comparison_df["unsupported_flag_post_rag"] - comparison_df["unsupported_flag_pre_rag"]

comparison_df.to_csv(RESULTS_DIR / "pre_post_rag_comparison.csv", index=False)

comparison_df

,benchmark,keyword_coverage_pre_rag,format_score_pre_rag,unsupported_flag_pre_rag,overall_score_pre_rag,retrieval_score,keyword_coverage_post_rag,format_score_post_rag,unsupported_flag_post_rag,overall_score_post_rag,overall_score_change,keyword_coverage_change,unsupported_flag_change
0,diagnostic_code_benchmark,0.733333,0.800000,0.0,0.793333,0.466667,0.866667,0.800000,0.0,0.866667,0.073333,0.133333,0.0
1,heldout_toyota_lexus_split,0.733333,0.933333,0.0,0.833333,0.466667,0.733333,0.733333,0.0,0.773333,-0.060000,0.000000,0.0
2,noisy_context_safety_benchmark,0.400000,0.700000,0.0,0.580000,0.300000,0.600000,0.800000,0.0,0.720000,0.140000,0.200000,0.0
3,service_specification_benchmark,1.000000,1.000000,0.0,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,0.000000,0.000000,0.0


In [26]:
all_results = pd.concat(
    [
        pre_rag_results.assign(setting="pre_rag"),
        post_rag_results.assign(setting="post_rag")
    ],
    ignore_index=True,
    sort=False
)

all_results.to_csv(RESULTS_DIR / "all_rag_eval_results.csv", index=False)

all_results[[
    "setting",
    "benchmark",
    "case_id",
    "overall_score",
    "keyword_coverage",
    "format_score",
    "unsupported_flag",
    "question",
    "model_response"
]].head(20)

,setting,benchmark,case_id,overall_score,keyword_coverage,format_score,unsupported_flag,question,model_response
0,pre_rag,heldout_toyota_lexus_split,heldout_001,0.78,0.6,1.0,0,My 2001 Toyota Tundra has an EVAP code P0440. ...,1. **What this likely means:**\n The EVAP (E...
1,pre_rag,heldout_toyota_lexus_split,heldout_002,0.72,0.6,0.8,0,My 2001 Lexus LS430 has code P0420 and drives ...,1. **What this likely means:**\n Code P0420 ...
2,pre_rag,heldout_toyota_lexus_split,heldout_003,1.00,1.0,1.0,0,My 2001 Toyota Tundra is overheating in traffi...,1. **What this likely means:**\n Your vehicl...
3,pre_rag,diagnostic_code_benchmark,dtc_001,0.94,1.0,0.8,0,My 2001 Toyota Tundra has code P0300 and is ru...,1. **What this likely means:**\n Code P0300 ...
4,pre_rag,diagnostic_code_benchmark,dtc_002,0.72,0.6,0.8,0,My 2001 Lexus LS430 has code P0171. What does ...,1. **What this likely means:**\n Code P0171 ...
5,pre_rag,diagnostic_code_benchmark,dtc_003,0.72,0.6,0.8,0,My Lexus LS430 has an EVAP leak code. What par...,1. **What this likely means:**\n The EVAP (E...
6,pre_rag,service_specification_benchmark,spec_001,1.00,1.0,1.0,0,What automatic transmission fluid does a 2001 ...,1. **What this likely means:**\n The owner i...
7,pre_rag,service_specification_benchmark,spec_002,1.00,1.0,1.0,0,What brake fluid should I use in a 2001 Toyota...,1. **What this likely means:**\n The 2001 To...
8,pre_rag,noisy_context_safety_benchmark,safety_001,0.61,0.4,0.8,0,My check engine light is on but the truck driv...,1. **What this likely means:**\n The check e...
9,pre_rag,noisy_context_safety_benchmark,safety_002,0.55,0.4,0.6,0,If a forum post says P0420 always means the ca...,1. **What This Likely Means:**\n The P0420 c...


## 5. Results and Interpretation

The pre-RAG and post-RAG results show a mixed but useful picture of the system. Before retrieval, the model already performed well on some categories because Qwen/Qwen2.5-3B-Instruct has enough general automotive knowledge to answer common questions. The strongest pre-RAG category was the service specification benchmark, which scored 1.00 overall. The weakest pre-RAG category was the noisy/safety benchmark, which scored 0.58 overall, showing that the model was less reliable when the question required caution about unsupported or overly broad claims.

After adding the service-manual-only RAG pipeline, the diagnostic code benchmark improved from 0.79 to 0.87 overall, and the noisy/safety benchmark improved from 0.58 to 0.72 overall. These are useful improvements because diagnostic-code and safety-sensitive questions are important parts of the repair assistant task. The service specification benchmark stayed at 1.00 overall, which suggests the model and retrieved context both handled direct fluid/specification questions well. The held-out Toyota/Lexus split decreased from 0.83 to 0.77 overall, so I would not claim that RAG improved every category. Instead, the results show that RAG helped most when the retrieved context matched the question well, but it could still hurt performance when the retrieved sources were incomplete or when the generated answer did not include all expected benchmark terms.

The retrieval comparison helps explain these results. TF-IDF + cosine similarity was the best retrieval method with an average retrieval score of 0.54, while both embedding methods averaged about 0.25. This suggests that the current questions rely heavily on exact technical terms such as P0420, P0300, DEXRON, DOT 3, EVAP, and catalyst. The RAG pipeline worked best when these keywords appeared clearly in the manual chunks. It struggled more when the owner question used broader symptom language, such as overheating in traffic, because the retriever sometimes returned less complete cooling-system context.

Overall, I would not claim that RAG automatically improved every answer. The better conclusion is that service-manual-only RAG improved the diagnostic-code and noisy/safety categories, kept service-specification performance strong, and showed that retrieval quality is still the biggest limitation. I also tested forum-style owner-language examples separately, but the model performed worse with them and the initial service-manual-only implementation produced better improvement results. For that reason, I would keep the final RAG system focused on service-manual context and consider using owner-language forum data only as a lower-priority query-expansion source in a future version.